# Characterize nodules at their ground-truth locations with MedGemma

Runs `5b_characterize_ground_truth_nodules.py`: for every pylidc consensus nodule already recorded in `ground_truth_annotations.json` (patients 1-20), crops directly around that nodule's own real centroid/diameter (no detector involved) and asks MedGemma 1.5 to rate it on the same 9 pylidc attributes as `5_characterize_nodules.py` - then compares straight against the mean of that nodule's own radiologist annotations. No MONAI/simpleitk needed, since there's no detector step.

Before running anything:
1. **Runtime > Change runtime type > GPU** (T4 is fine).
2. Upload `lidc_idri_p1-20.zip` to your Google Drive (same zip used for the counting notebook - patients 1-20 + `annotations.csv`, ~1.1GB). Skip this if you already have it there from before.
3. Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) holding a Hugging Face access token that has accepted the MedGemma license at https://huggingface.co/google/medgemma-1.5-4b-it.

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
!git clone https://github.com/freya-gul/rail.git
%cd rail
!git checkout medgemma-characterization-variants

Cloning into 'rail'...
remote: Enumerating objects: 165, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 165 (delta 79), reused 125 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (165/165), 74.34 MiB | 3.47 MiB/s, done.
Resolving deltas: 100% (79/79), done.
Updating files: 100% (54/54), done.
Encountered 2 files that should have been pointers, but weren't:
	image_download/monai_bundles/lung_nodule_ct_detection/models/model.pt
	image_download/monai_bundles/lung_nodule_ct_detection/models/model.ts
/content/rail/rail
error: Your local changes to the following files would be overwritten by checkout:
	image_download/monai_bundles/lung_nodule_ct_detection/models/model.pt
	image_download/monai_bundles/lung_nodule_ct_detection/models/model.ts
Please commit your changes or stash them before you switch branches.
Aborting


Point this at wherever you uploaded `lidc_idri_p1-20.zip` in Drive. It unzips into `datasets/LDIC-IDRI-subset/` inside the cloned repo — that's the path this script's `DICOM_ROOT` already expects:

In [13]:
ZIP_PATH = "/content/drive/MyDrive/lidc_idri_p1-20.zip"  # <-- update to your actual upload path
DATA_DIR = "datasets/LDIC-IDRI-subset"  # relative to the repo root (we've already %cd'd into rail)

import pathlib
assert pathlib.Path(ZIP_PATH).exists(), f"{ZIP_PATH} not found — check the path/upload"

In [ ]:
!mkdir -p {DATA_DIR}
!unzip -q {ZIP_PATH} -d {DATA_DIR}
!ls {DATA_DIR}

In [15]:
# No monai/simpleitk needed - this script never touches the MONAI detector, only
# crops directly out of the raw DICOM series at the ground-truth nodule locations.
!pip install -q pydicom "transformers>=5.12.1" "huggingface_hub>=1.21.0"

In [16]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

Sanity check: GPU visible to torch (the script already defaults to `cuda` > `mps` > `cpu`):

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA available: True
NVIDIA A100-SXM4-40GB


## Run characterization

Prints live progress per nodule (predicted vs. ground-truth attribute values aren't shown inline, but pass/fail on JSON validation is) plus a running ETA. Resumable at the individual-nodule level — a killed run picks back up partway through a patient rather than redoing it.

In [8]:
!python image_download/5b_characterize_ground_truth_nodules.py --start 1 --end 20

Loading MedGemma 1.5 on cuda... (64 nodule(s) to characterize)
config.json: 100% 2.55k/2.55k [00:00<00:00, 7.37MB/s]
model.safetensors.index.json: 100% 90.6k/90.6k [00:00<00:00, 97.5MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/8.60G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  33% 2.82G/8.60G [00:09<00:19, 293MB/s,  103MB/s  ]
Reconstructing (incomplete total...):  50% 4.29G/8.60G [00:13<00:11, 376MB/s,  103MB/s  ]
Reconstructing (incomplete total...):  88% 7.60G/8.60G [00:21<00:02, 378MB/s,  187MB/s  ]
Reconstructing (incomplete total...): 100% 8.60G/8.60G [00:22<00:00, 465MB/s,  202MB/s  ]

Fetching 2 files: 100% 2/2 [00:22<00:00, 11.32s/it]
Download complete: 100% 7.79G/7.79G [00:22<00:00, 303MB/s,  220MB/s  ]
Download complete: 100% 7.79G/7.79G [00:22<00:00, 343MB/s,  220MB/s  ]
Reconstruction complete: 100% 8.60G/8.60G [00:22<00:00

## Peek at partial results anytime

Run this whenever you want — while the run above is still going, if it got interrupted, or once it's fully done. It reads whatever `nodule_characteristics_gt/<patient>.json` files already exist on disk and computes the same MAE/bias summary the final cell below does, over however many nodules have actually been characterized so far. No need to wait for all `--end` patients to finish, and safe to re-run repeatedly as more come in.

In [9]:
import json
import sys
from pathlib import Path

sys.path.insert(0, "image_download")
from lidc_attributes import LIDC_ATTRIBUTES

OUTPUT_DIR = Path("image_download/nodule_characteristics_gt")
patient_files = sorted(OUTPUT_DIR.glob("LIDC-IDRI-*.json"))
rows = [row for f in patient_files for row in json.loads(f.read_text())]

print(f"{len(rows)} nodule(s) characterized so far across {len(patient_files)} patient(s)\n")
print(f"{'attribute':<18}{'MAE':>8}{'bias':>8}{'n':>6}   scale")
for a in LIDC_ATTRIBUTES:
    errs = [r[f"{a}_err"] for r in rows if r.get(f"{a}_err") is not None]
    mae = sum(abs(e) for e in errs) / len(errs) if errs else None
    bias = sum(errs) / len(errs) if errs else None
    lo, hi = min(LIDC_ATTRIBUTES[a]["labels"]), max(LIDC_ATTRIBUTES[a]["labels"])
    mae_str = f"{mae:.2f}" if mae is not None else "n/a"
    bias_str = f"{bias:+.2f}" if bias is not None else "n/a"
    print(f"{a:<18}{mae_str:>8}{bias_str:>8}{len(errs):>6}   {lo}-{hi}")

45 nodule(s) characterized so far across 12 patient(s)

attribute              MAE    bias     n   scale
subtlety              1.04   -0.05    43   1-5
internalStructure     0.07   +0.07    43   1-4
calcification         0.51   +0.27    43   1-6
sphericity            0.84   -0.19    43   1-5
margin                0.90   -0.06    43   1-5
lobulation            0.92   +0.56    43   1-5
spiculation           0.53   -0.34    43   1-5
texture               0.58   +0.26    43   1-5
malignancy            0.97   +0.28    43   1-5


## Results

- `image_download/nodule_characteristics_gt/<patient>.json` — per-nodule predicted attributes, ground-truth mean, error, raw MedGemma response, and JSON-validation problems (if any).
- `image_download/characterize_ground_truth_comparison.csv` — the same data flattened across all patients, one row per nodule.
- `image_download/characterize_ground_truth_summary.json` — MAE and signed bias per attribute, aggregated across every nodule.

Quick look at the summary table:

In [18]:
import json
summary = json.load(open("image_download/characterize_ground_truth_summary.json"))
for attr, stats in summary.items():
    print(f"{attr:<18} MAE={stats['mae']:.2f}  bias={stats['bias']:+.2f}  n={stats['n']}")

FileNotFoundError: [Errno 2] No such file or directory: 'image_download/characterize_ground_truth_summary.json'

## Try prompt variants: anchored guidance / few-shot examples

Runs `5c_characterize_variants.py`, an A/B sibling of the script above. The first nodule characterized (`LIDC-IDRI-0001` #0) showed a near-worst-case spiculation miss (predicted 1 "No Spiculation" vs. ground truth 4.25 "Marked Spiculation") that matches a bias `bias_correction.json` already measured on the detector-based pipeline (subtlety -0.53, margin -0.55, spiculation -0.44). Two cheap levers to test before reaching for LoRA fine-tuning:

- `--anchored` — adds targeted anti-underrating guidance to the subtlety/margin/spiculation attribute descriptions. Free (a few extra sentences, no extra images).
- `--fewshot` — prepends two fixed calibration examples (real images + their consensus-rounded ground truth) as prior conversation turns: one unambiguous low-spiculation nodule and one unambiguous high-spiculation nodule, both 4/4-reader consensus. Costs roughly 2x the vision tokens/time per nodule.

Combinable, and each flag combination writes to its own directory (`nodule_characteristics_gt_anchored/`, `..._fewshot/`, `..._anchored_fewshot/`) so nothing clobbers the zero-shot baseline above. Try a small `--end` first (a handful of patients) before committing to the full 20 — few-shot in particular roughly doubles per-nodule runtime.

In [19]:
!python image_download/5c_characterize_variants.py --anchored --start 1 --end 5

Variant: anchored=True fewshot=False -> /content/rail/rail/image_download/nodule_characteristics_gt_anchored
Loading MedGemma 1.5 on cuda... (10 nodule(s) to characterize)
Loading weights: 100% 883/883 [00:00<00:00, 5777.45it/s]
[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'return_dict_in_generate'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[1/5] LIDC-IDRI-0001 nodule 0 (43.0s): OK  ETA 6m27s (9 nodule(s) left)
[1/5] 

In [20]:
!python image_download/5c_characterize_variants.py --fewshot --start 1 --end 5

Variant: anchored=False fewshot=True -> /content/rail/rail/image_download/nodule_characteristics_gt_fewshot
Loading MedGemma 1.5 on cuda... (9 nodule(s) to characterize)
Loading weights: 100% 883/883 [00:00<00:00, 4885.04it/s]
Cropping few-shot calibration examples...
[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.
[transformers] Passing `generation_config` together with generation-related arguments=({'return_dict_in_generate', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[1/5] LIDC-IDRI-0001 nodule 0 (8.6s): 

In [21]:
!python image_download/5c_characterize_variants.py --anchored --fewshot --start 1 --end 5

Variant: anchored=True fewshot=True -> /content/rail/rail/image_download/nodule_characteristics_gt_anchored_fewshot
Loading MedGemma 1.5 on cuda... (9 nodule(s) to characterize)
Loading weights: 100% 883/883 [00:00<00:00, 5334.01it/s]
Cropping few-shot calibration examples...
[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'return_dict_in_generate'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[1/5] LIDC-IDRI-0001 nodule 0 

Compare all variants (plus the zero-shot baseline from above) side by side, over whatever patients each has completed so far — safe to re-run any time, doesn't require any of them to be finished:

In [22]:
import importlib.util
from pathlib import Path

spec = importlib.util.spec_from_file_location("variants", "image_download/5c_characterize_variants.py")
variants_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(variants_mod)

variants_mod.compare_variants({
    "baseline": Path("image_download/nodule_characteristics_gt"),
    "anchored": Path("image_download/nodule_characteristics_gt_anchored"),
    "fewshot": Path("image_download/nodule_characteristics_gt_fewshot"),
    "anchored+fewshot": Path("image_download/nodule_characteristics_gt_anchored_fewshot"),
})

attribute             baseline MAE   baseline bias    anchored MAE   anchored bias     fewshot MAE    fewshot biasanchored+fewshot MAEanchored+fewshot bias
subtlety                       n/a             n/a            1.20           +0.80            1.50           +1.50            1.72           +1.72
internalStructure              n/a             n/a            0.00           +0.00            0.00           +0.00            0.00           +0.00
calcification                  n/a             n/a            0.23           +0.23            0.25           +0.25            0.25           +0.25
sphericity                     n/a             n/a            1.10           -0.75            0.47           +0.03            0.47           +0.03
margin                         n/a             n/a            1.15           +0.45            1.28           -0.72            1.28           -0.72
lobulation                     n/a             n/a            1.20           +0.80            1.11           

{'baseline': [],
 'anchored': [{'patient_id': 'LIDC-IDRI-0001',
   'nodule_index': 0,
   'num_annotations': 4,
   'diameter_mm': 32.76,
   'validation_problems': [],
   'raw_response': '<unused94>thought\nThe user wants me to analyze a sequence of CT slices showing a lung nodule and provide a single JSON object containing 9 characteristics of the nodule, each scored on a 1-5 scale.\n\n1.  **Analyze the images:** The images show a nodule in the lung. It appears somewhat round/ovoid, with a relatively sharp margin in most slices. It has a somewhat solid texture, possibly with some ground glass components, but mostly dense. There are no obvious calcifications. It doesn\'t appear highly spiculated, but there might be some subtle radiating lines in a few slices. It\'s not clearly lobulated. It\'s not extremely subtle, nor is it obviously obvious.\n\n2.  **Score each characteristic:**\n    *   **subtlety:** The nodule is clearly visible and denser than the surrounding lung parenchyma. It\'s 

## Save Results to Google Drive

The files generated in the Colab environment are temporary and will be deleted when the runtime disconnects. To save your results permanently, you can copy them to your mounted Google Drive.

In [ ]:
import shutil

source_file = "/content/rail/image_download/nodule_characteristics_gt/LIDC-IDRI-0020.json"
destination_path = "/content/drive/MyDrive/Colab_MedGemma_Results/"

# Create the destination directory if it doesn't exist
import os
os.makedirs(destination_path, exist_ok=True)

# Copy the file
shutil.copy(source_file, destination_path)
print(f"File copied to: {destination_path}")

## Push to GitHub

To push changes to GitHub, you need to configure your Git identity and provide authentication. It is recommended to use a Personal Access Token (PAT) stored as a Colab secret for security.

First, make sure you have generated a GitHub PAT with `repo` permissions and saved it as a Colab secret named `GH_TOKEN`.

In [ ]:
!git lfs uninstall

In [ ]:
!git status

In [ ]:
!printf 'image_download/monai_bundles/lung_nodule_ct_detection/models/* -filter -diff -merge -text\n' >> .git/info/attributes
%env GIT_LFS_SKIP_SMUDGE=1

In [ ]:
!git checkout -- image_download/monai_bundles/lung_nodule_ct_detection/models/model.ts
!git status

In [ ]:
!git add image_download/characterize_ground_truth_comparison.csv image_download/characterize_ground_truth_summary.json
!git commit -m "Add characterize_ground_truth summary/comparison"

In [ ]:
!git status
!git config pull.rebase true
!git pull

In [ ]:
!git push origin medgemma-characterization-variants
